# Prototype Analysis on Sample

Dieses Notebook testet erste Analyseideen auf einem kleinen Sample der bereinigten Parking-Violations-Daten.

Ziel:
- processed Parquet-Daten aus HDFS laden
- kleines Sample erstellen
- Analysefragen testen
- prüfen, welche Queries und Charts für die finale Analyse sinnvoll sind

Die finalen Analysen werden später in `src/4_Analysis` auf dem vollständigen Datensatz ausgeführt.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc

spark = SparkSession.builder \
    .appName("BDLC_Parking_Violations_Prototype") \
    .master("spark://bdlc-012.bdlc.ls.eee.intern:7077") \
    .config("spark.executor.cores", "4") \
    .config("spark.executor.memory", "8g") \
    .config("spark.cores.max", "4") \
    .getOrCreate()

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/09 20:50:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/09 20:50:40 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/05/09 20:50:40 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


26/05/09 20:50:55 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [3]:
processed_path = "hdfs:///parking_violations/processed/parking_violations_cleaned"

df = spark.read.parquet(processed_path)

df.groupBy("fiscal_year").count().orderBy("fiscal_year").show()

+-----------+--------+
|fiscal_year|   count|
+-----------+--------+
|       2023|21563238|
|       2024|16099641|
|       2025|16557773|
+-----------+--------+



In [4]:
sample_df = df.sample(fraction=0.01, seed=42)

sample_count = sample_df.count()
sample_count

542884

In [5]:
sample_df.groupBy("violation_code", "violation_description") \
    .count() \
    .orderBy(desc("count")) \
    .show(20, truncate=False)

[Stage 7:=====================================>                    (9 + 4) / 14]

+--------------+------------------------------+------+
|violation_code|violation_description         |count |
+--------------+------------------------------+------+
|36            |PHTO SCHOOL ZN SPEED VIOLATION|186542|
|21            |21-No Parking (street clean)  |45830 |
|38            |38-Failure to Dsplay Meter Rec|37503 |
|14            |14-No Standing                |25784 |
|5             |BUS LANE VIOLATION            |23122 |
|7             |FAILURE TO STOP AT RED LIGHT  |22937 |
|40            |40-Fire Hydrant               |19381 |
|21            |No Parking Street Cleaning    |16764 |
|71            |71A-Insp Sticker Expired (NYS)|16166 |
|20            |20A-No Parking (Non-COM)      |14448 |
|70            |70A-Reg. Sticker Expired (NYS)|10432 |
|37            |37-Expired Parking Meter      |8507  |
|31            |31-No Stand (Com. Mtr. Zone)  |8344  |
|69            |69-Fail to Dsp Prking Mtr Rcpt|7563  |
|19            |19-No Stand (bus stop)        |7351  |
|16       

In [6]:
sample_df.groupBy("violation_code") \
    .count() \
    .orderBy(desc("count")) \
    .show(20, truncate=False)

[Stage 10:================================================>       (12 + 2) / 14]

+--------------+------+
|violation_code|count |
+--------------+------+
|36            |186542|
|21            |62827 |
|38            |37559 |
|14            |27299 |
|5             |23122 |
|7             |22937 |
|40            |21652 |
|20            |20797 |
|71            |19074 |
|70            |12858 |
|46            |11108 |
|37            |8513  |
|31            |8375  |
|19            |7830  |
|74            |7737  |
|69            |7563  |
|16            |7492  |
|12            |5934  |
|43            |5303  |
|15            |4949  |
+--------------+------+
only showing top 20 rows



In [7]:
sample_df.groupBy("vehicle_make") \
    .count() \
    .orderBy(desc("count")) \
    .show(20, truncate=False)

+------------+-----+
|vehicle_make|count|
+------------+-----+
|HONDA       |64431|
|TOYOT       |63846|
|FORD        |50321|
|NISSA       |42201|
|CHEVR       |29271|
|ME/BE       |28161|
|BMW         |26654|
|JEEP        |24832|
|HYUND       |18971|
|LEXUS       |13427|
|ACURA       |12138|
|FRUEH       |12087|
|SUBAR       |11549|
|KIA         |11478|
|DODGE       |10758|
|AUDI        |10197|
|VOLKS       |10038|
|MAZDA       |9916 |
|RAM         |8174 |
|INFIN       |8123 |
+------------+-----+
only showing top 20 rows



In [8]:
sample_df.groupBy("fiscal_year", "issue_month") \
    .count() \
    .orderBy("fiscal_year", "issue_month") \
    .show(50)

+-----------+-----------+-----+
|fiscal_year|issue_month|count|
+-----------+-----------+-----+
|       2023|          1|13404|
|       2023|          2|12671|
|       2023|          3|15228|
|       2023|          4|13737|
|       2023|          5|15146|
|       2023|          6|16494|
|       2023|          7|28691|
|       2023|          8|32370|
|       2023|          9|25028|
|       2023|         10|15147|
|       2023|         11|14538|
|       2023|         12|12526|
|       2024|          1|12277|
|       2024|          2|12561|
|       2024|          3|13444|
|       2024|          4|13214|
|       2024|          5|14471|
|       2024|          6|13283|
|       2024|          7|15087|
|       2024|          8|14860|
|       2024|          9|12666|
|       2024|         10|14203|
|       2024|         11|13769|
|       2024|         12|11878|
|       2025|          1|12103|
|       2025|          2|11765|
|       2025|          3|14544|
|       2025|          4|14819|
|       

In [9]:
sample_df.groupBy("issue_weekday") \
    .count() \
    .orderBy("issue_weekday") \
    .show()

+-------------+-----+
|issue_weekday|count|
+-------------+-----+
|            1|53236|
|            2|78477|
|            3|87064|
|            4|81344|
|            5|88397|
|            6|85925|
|            7|68441|
+-------------+-----+



## Erkenntnisse aus dem Prototyping

Das 1%-Sample enthält 542'884 Zeilen und ist damit gross genug, um Analyseideen zu testen.

Getestete Analysefragen:

1. **Welche Violation Codes kommen am häufigsten vor?**  
   Diese Analyse ist sinnvoll. `violation_code = 36` ist im Sample mit Abstand am häufigsten. Für die finale Analyse sollte primär nach `violation_code` gruppiert werden, da einzelne Codes unterschiedliche Beschreibungen haben können.

2. **Welche Vehicle Makes erhalten am häufigsten Parking Violations?**  
   Diese Analyse ist sinnvoll. Im Sample sind HONDA, TOYOT, FORD und NISSA besonders häufig.

3. **Gibt es zeitliche Muster nach Monat?**  
   Diese Analyse ist sinnvoll. Besonders FY2023 zeigt auffällige Werte in den Monaten Juli, August und September.

4. **Gibt es zeitliche Muster nach Wochentag?**  
   Diese Analyse ist sinnvoll. Sonntag weist deutlich weniger Verstösse auf als die Werktage.

Die getesteten Analysen werden im nächsten Schritt in `src/4_Analysis` auf dem vollständigen Datensatz ausgeführt.